In [1]:
import pandas as pd
import os

In [3]:
SAMPLE_DIR = r"D:\projects\Healthcare\data\raw_sample"
df = pd.read_csv(os.path.join(SAMPLE_DIR, "stroke_sliced.csv"))

In [4]:
#Missing value detection

missing = df.isna().sum()
missing_pct = (df.isna().mean()*100).round(2)

In [6]:
#only show columns that actually have missing values
missing_report = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct
})
missing_report = missing_report[missing_report["missing_count"] > 0]

print("columns with missing values:\n")
print(missing_report)
print(f"\nTotal missing cells: {df.isna().sum().sum()}")

columns with missing values:

          missing_count  missing_pct
Symptoms           2500        16.67

Total missing cells: 2500


In [13]:
def treat_missing(df, drop_threshold=0.5):
    """Columns with missing fraction above drop_threshold are dropped."""
    df = df.copy()
    report = {"dropped_columns": [], "filled_columns":{}}

    for col in df.columns:
        miss_frac = df[col].isna().mean()

        if miss_frac == 0:
            continue

        if miss_frac > drop_threshold:
            df = df.drop(columns=[col])
            report["dropped_columns"].append(col)
            continue
        if pd.api.types.is_numeric_dtype(df[col]):
            fill_value = df[col].median()
            method = "median"
        else:
            fill_value = df[col].mode()[0]
            method = "mode"
        df[col] = df[col].fillna(fill_value)
        report["filled_columns"][col] = {
                "method": method,
                "fill_value": fill_value,
                "pct_filled": round(miss_frac *100,2),
        }
        return df, report
    
df_clean, missing_report = treat_missing(df)

print("Dropped columns:", missing_report["dropped_columns"])
print("\nFilled columns:")
for col, info in missing_report["filled_columns"].items():
     print(f"   {col:20} method={info['method']:8} value={info['fill_value']} ({info['pct_filled']}%filled)")
print(f"\nMissing after treatment: {df_clean.isna().sum().sum()}")






Dropped columns: []

Filled columns:
   Symptoms             method=mode     value=Difficulty Speaking (16.67%filled)

Missing after treatment: 0


Duplicate detection and removal

In [17]:
def remove_duplicates(df):

    df = df.copy()

    n_before = df.shape[0]
    n_duplicates = int(df.duplicated().sum())

    df = df.drop_duplicates()
    df = df.reset_index(drop=True)

    n_after = df.shape[0]

    report = {
        "rows_before": n_before,
        "duplicates_found": n_duplicates,
        "rows_after": n_after,
        "rows_removed": n_before - n_after,
    }
    return df, report
#Test
df_nodup, dup_report = remove_duplicates(df_clean)

print("Duplicate removal report:")
for key, value in dup_report.items():
    print(f" {key:<20}: {value}")

Duplicate removal report:
 rows_before         : 15000
 duplicates_found    : 0
 rows_after          : 15000
 rows_removed        : 0


Outlier detection and capping

In [23]:
def treat_outliers(df, factor=1.5):

    df = df.copy()
    report = {}

    numeric_cols = df.select_dtypes(include="number").columns

    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - factor * IQR
        upper = Q3 + factor * IQR

        #count outliers before capping

        n_outliers = int(((df[col] < lower) | (df[col] > upper)).sum())

        if n_outliers > 0:
            df[col] = df[col].clip(lower=lower, upper=upper)
            report[col] = {
                "outliers": n_outliers,
                "lower_bound": round(lower, 2),
                "upper_bound": round(upper, 2),
            }
        return df, report
    
#Test
df_final, outlier_report = treat_outliers(df_nodup)

print("Outlier treatment report:\n")
if outlier_report:
    for col, info in outlier_report.items():
        print(f"  {col:<25} outliers={info['outliers']:<6}"
              f"bounds=[{info['lower_bound']}, {info['upper_bound']}]")
else:
    print("No outliers found")

print(f"\nFinal shape: {df_final.shape}")

Outlier treatment report:

No outliers found

Final shape: (15000, 22)
